# RECAP: Bind/Unbind Roundtrip Test

Verifies that `module.bind(merged_params).heads["action"].predict_action` preserves LoRA-modified weights through the `unbind()` call inside `DiffusionActionHead.predict_action`.

**Run this on Colab (T4 GPU) or any machine with a GPU.**

4 tests:
1. LoRA changes action head output (bind with merged ≠ bind with original)
2. Bind approach matches replace approach (unbind roundtrip preserves params)
3. Different RNG seeds → different actions (diffusion sampling works)
4. Transformer output unchanged by LoRA (LoRA only touches action head)

In [ ]:
# ── Setup: install deps + clone repo ──────────────────────────────
# Skip this cell if already installed (e.g. Baseten interactive)

!pip install -q "jax[cuda12_pip]==0.4.20" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
!pip install -q flax==0.7.5 optax==0.1.5 orbax-checkpoint==0.5.3 tensorstore==0.1.51
!pip install -q tensorflow==2.15.0 tensorflow_datasets==4.9.2
!pip install -q distrax==0.1.5 einops transformers==4.36.2 huggingface_hub
!pip install -q numpy==1.24.3

# Clone TTDR (skip if already present)
import os
if not os.path.exists('TTDR'):
    !git clone https://github.com/YOUR_REPO/TTDR.git  # TODO: set your repo URL
    %cd TTDR
    !pip install -e .
else:
    %cd TTDR

In [ ]:
# ── HuggingFace auth (needed for private repos) ──────────────────
# The world model checkpoint at 4manifold/ttdr-world-model is private.
# You need a HF token with read access.

from huggingface_hub import login
# Option 1: interactive prompt
login()
# Option 2: paste token directly (don't commit this!)
# login(token="hf_...")

In [ ]:
# ── Check environment ─────────────────────────────────────────────
import jax
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")
print(f"Backend: {jax.default_backend()}")
assert jax.default_backend() != 'cpu', "No GPU found! This will be very slow."

In [ ]:
# ── Load Octo ─────────────────────────────────────────────────────
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  # don't let TF grab GPU memory

from octo.model.octo_model import OctoModel

print("Loading Octo (downloads ~1.5GB on first run)...")
octo_model = OctoModel.load_pretrained("hf://rail-berkeley/octo-base-1.5")
octo_params = octo_model.params
print("Loaded.")

In [ ]:
# ── Create dummy inputs ───────────────────────────────────────────
import jax.numpy as jnp

# Use Octo's own example batch (correct shapes guaranteed)
example_obs = jax.tree.map(lambda x: x[:1], octo_model.example_batch["observation"])
task = octo_model.create_tasks(texts=["pick up the object"])
pad_mask = example_obs["timestep_pad_mask"]

rng = jax.random.PRNGKey(42)
print(f"Obs keys: {list(example_obs.keys())}")
print(f"Obs image shape: {example_obs.get('image_primary', example_obs.get('image', next(v for k,v in example_obs.items() if 'image' in k))).shape}")

In [ ]:
# ── Create LoRA and merge ─────────────────────────────────────────
from recap.models.lora_adapter import init_lora_params, apply_lora

rng, lora_rng = jax.random.split(rng)
lora_params = init_lora_params(lora_rng, octo_params, rank=8)

print(f"LoRA targets {len(lora_params['layers'])} Dense layers:")
for path in list(lora_params['layers'].keys())[:3]:
    ab = lora_params['layers'][path]
    print(f"  {path}: A={ab['A'].shape}, B={ab['B'].shape}")
print(f"  ...")

# Make B non-zero so LoRA actually changes weights
# (normally B starts as zeros = no-op, only becomes non-zero after training)
for path, ab in lora_params["layers"].items():
    rng, key = jax.random.split(rng)
    lora_params["layers"][path]["B"] = jax.random.normal(key, ab["B"].shape) * 0.1

merged_params = apply_lora(octo_params, lora_params)
print("\nLoRA merged (B set to random for testing).")

In [ ]:
# ── Run transformer once (shared) ─────────────────────────────────
print("Running transformer...")
trans_out = octo_model.module.apply(
    {"params": octo_params},
    example_obs, task, pad_mask,
    train=False,
    method="octo_transformer",
)
print(f"Readout tokens shape: {trans_out['readout_action'].tokens.shape}")
print("Done.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TEST 1: LoRA changes action head output
# ══════════════════════════════════════════════════════════════════
rng, act_rng = jax.random.split(rng)

# Original params
action_head_orig = octo_model.module.bind({"params": octo_params}).heads["action"]
actions_orig = action_head_orig.predict_action(trans_out, rng=act_rng, train=False)

# Merged params (LoRA applied)
action_head_merged = octo_model.module.bind({"params": merged_params}).heads["action"]
actions_merged = action_head_merged.predict_action(trans_out, rng=act_rng, train=False)

diff = float(jnp.max(jnp.abs(actions_orig - actions_merged)))
print(f"TEST 1: bind(orig) vs bind(merged)")
print(f"  Max abs diff: {diff:.6f}")
print(f"  Actions orig:   {actions_orig[0, 0, :4]}")
print(f"  Actions merged: {actions_merged[0, 0, :4]}")
assert diff > 1e-4, f"FAIL: LoRA not taking effect! diff={diff}"
print(f"  ✓ PASS: LoRA changes action head output")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TEST 2: bind approach matches replace approach
# This is the critical test. If unbind() inside predict_action
# loses the LoRA params, these will differ.
# ══════════════════════════════════════════════════════════════════
merged_model = octo_model.replace(params=merged_params)
actions_replace = merged_model.sample_actions(example_obs, task, rng=act_rng)

diff = float(jnp.max(jnp.abs(actions_merged - actions_replace)))
print(f"TEST 2: bind approach vs replace approach")
print(f"  Max abs diff: {diff:.6f}")
print(f"  Actions bind:    {actions_merged[0, 0, :4]}")
print(f"  Actions replace: {actions_replace[0, 0, :4]}")
if diff < 1e-4:
    print(f"  ✓ PASS: bind matches replace — unbind roundtrip preserves params")
    print(f"  → Safe to split transformer + action head for efficiency")
else:
    print(f"  ✗ FAIL: bind differs from replace by {diff:.6f}")
    print(f"  → Must use octo_model.replace(params=merged).sample_actions() instead")
    print(f"  → This costs K extra transformer passes but is correct")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TEST 3: different RNG seeds produce different actions
# (confirms diffusion sampling randomness works)
# ══════════════════════════════════════════════════════════════════
rng, act_rng_2 = jax.random.split(rng)
actions_diff_rng = action_head_merged.predict_action(trans_out, rng=act_rng_2, train=False)

diff = float(jnp.max(jnp.abs(actions_merged - actions_diff_rng)))
print(f"TEST 3: same params, different RNG")
print(f"  Max abs diff: {diff:.6f}")
assert diff > 1e-4, f"FAIL: different RNG produced same actions! diff={diff}"
print(f"  ✓ PASS: K=3 sampling will produce K different action chunks")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# TEST 4: transformer output invariant to LoRA
# (LoRA only touches action head Dense layers in reverse_network)
# ══════════════════════════════════════════════════════════════════
trans_out_merged = octo_model.module.apply(
    {"params": merged_params},
    example_obs, task, pad_mask,
    train=False,
    method="octo_transformer",
)

tokens_orig = trans_out["readout_action"].tokens
tokens_merged = trans_out_merged["readout_action"].tokens
diff = float(jnp.max(jnp.abs(tokens_orig - tokens_merged)))
print(f"TEST 4: transformer invariance to LoRA")
print(f"  Max abs diff in readout tokens: {diff:.6f}")
assert diff < 1e-5, f"FAIL: LoRA leaked into transformer! diff={diff}"
print(f"  ✓ PASS: safe to run transformer once, reuse for both WM and action head")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════════════
print("="*60)
print("All tests passed.")
print()
print("Confirmed:")
print("  1. module.bind(merged).heads['action'].predict_action")
print("     preserves LoRA params through unbind() roundtrip")
print("  2. Transformer output identical for frozen vs LoRA params")
print("  3. Different RNG seeds → different diffusion samples")
print()
print("This means recap_adaptation.py can safely:")
print("  - Run transformer ONCE (batched CFG) with frozen params")
print("  - Run action head K times with merged params (no re-transformer)")
print("  - 2 transformer passes per step instead of 2+2K")